# 1. Converting PDF into Docling Document

In [1]:
from docling.document_converter import DocumentConverter

source = r"my_papers/2024 - 15.pdf"  # file path or URL
converter = DocumentConverter()
doc = converter.convert(source).document

#print(doc.export_to_markdown())  # output: "### Docling Technical Report[...]" 

C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-02-11 12:35:38,534 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-11 12:35:38,565 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-11 12:35:38,583 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-11 12:35:38,585 [RapidOCR] main.py:50: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCR

In [2]:
from IPython.display import Markdown, display

#display(Markdown(doc.export_to_markdown()))



**Docling Serialisation** is the method which can allow to add text descriptions or actual images at their respective locations. This approach can be explored if need for such enrichment rise. 

As of now, focus is to prototype a basic text-centric workflow.

# 2. Chunking

Docling has two method for chunking **Base Chunker** and **Hybrid Chunker**. Base chunker is the blueprint to be followed for custom chunking strategies implementation, if needed. Hybrid chunker already provides *"Hybrid chunking applies tokenization-aware refinements on top of document-based hierarchical chunking."* 

As of now, Hybrid chunker is ideal for this workflow. 

**Configuring Tokenisation**

Use same model for chunker and embedding.

In [3]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

from docling.chunking import HybridChunker

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 128  # set to a small number for illustrative purposes

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,  # optional, by default derived from `tokenizer` for HF case
)

Instantiate the Chunker

In [4]:
chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,  # optional, defaults to True
)
chunk_iter = chunker.chunk(dl_doc=doc)
chunks = list(chunk_iter)

Token indices sequence length is longer than the specified maximum sequence length for this model (1723 > 512). Running this sequence through the model will result in indexing errors


*ignore the warning!* [more info](https://docling-project.github.io/docling/faq/#hybridchunker-triggers-warning-token-indices-sequence-length-is-longer-than-the-specified-maximum-sequence-length-for-this-model)

In [ ]:
print(f'Total chunks: {len(chunks)}\n')

# Example: print the first chunk's text content
print(f'Example text from chunk: \n\n{chunks[5].text}')

print(f'\nExample metadata from chunk: \n\n{chunks[5].meta}')

In [ ]:
JSON_Data = chunks[5].meta.export_json_dict()

JSON_Data




**Contextualise**

It joins joins some words to keep the contextual meaning. For e.g., look 2,3, author names are attached with affiliation. 

In [5]:
for i, chunk in enumerate(chunks):
    print(f"=== {i} ===")
    txt_tokens = tokenizer.count_tokens(chunk.text)
    print(f"chunk.text ({txt_tokens} tokens):\n{chunk.text!r}")

    ser_txt = chunker.contextualize(chunk=chunk)
    ser_tokens = tokenizer.count_tokens(ser_txt)
    print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_txt!r}")


=== 0 ===
chunk.text (83 tokens):
'JHTT 15,4\n592\nReceived22 June 2023 Revised 11 September 2023 21 November 2023 7 February 2024 Accepted16 April 2024\nJournal of Hospitality and Tourism Technology Vol. 15 No. 4, 2024 pp. 592-609\n©EmeraldPublishingLimited 1757-9880 DOI 10.1108/JHTT-06-2023-0170'
chunker.contextualize(chunk) (83 tokens):
'JHTT 15,4\n592\nReceived22 June 2023 Revised 11 September 2023 21 November 2023 7 February 2024 Accepted16 April 2024\nJournal of Hospitality and Tourism Technology Vol. 15 No. 4, 2024 pp. 592-609\n©EmeraldPublishingLimited 1757-9880 DOI 10.1108/JHTT-06-2023-0170'
=== 1 ===
chunk.text (23 tokens):
'Xinquan Cheng\nDepartment of Tourism Management, Pai Chai University, Daejeon, Republic of Korea'
chunker.contextualize(chunk) (55 tokens):
"Anovel ChatGPT-based multimodel framework for tourism review mining: a case study on China ' s /uniFB01 ve sacred mountains\nXinquan Cheng\nDepartment of Tourism Management, Pai Chai University, Daejeon, Republic of 

### Advanced Serialisation and Chunking

Source: [Advanced Chunking and Serialisation](https://docling-project.github.io/docling/examples/advanced_chunking_and_serialization/#using-a-custom-strategy)

It will be used to add description (or something more) at places of Tables and Figures/Pictures. 



**Table Serialisation**

In [6]:
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer


class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=MarkdownTableSerializer(),  # configuring a different table serializer
        )


chunker = HybridChunker(
    tokenizer=tokenizer,
    serializer_provider=MDTableSerializerProvider(),
)

chunk_iter = chunker.chunk(dl_doc=doc)

chunks = list(chunk_iter)


**Picture Serialisation**

Look at it later, it will add image descriptions at image locations. [Look Here](https://docling-project.github.io/docling/examples/advanced_chunking_and_serialization/#using-a-custom-strategy)